# Compare This Repository With the Official Paper Code

This notebook is intentionally light. It does not merge the two pipelines into one wrapper. Instead, it prepares both codebases in the same Colab runtime so you can run this repository and the official notebook-based implementation under comparable seeds and compute budgets.

## 1. Clone This Repository With the Official Submodule

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main"
REPO_DIR = Path("/content/rea_of_llms_supplementary_code-main")

if REPO_DIR.exists():
    print(f"Repository already exists: {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive"], check=True)
else:
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, str(REPO_DIR)], check=True)

OFFICIAL_DIR = REPO_DIR / "external" / "rea_of_llms_official"
print("Local repo:", REPO_DIR)
print("Official paper code:", OFFICIAL_DIR)

## 2. Install Shared Dependencies

The official repository uses notebooks and its own `pyproject.toml`. In Colab, this notebook installs the dependency set from this repository to avoid forcing the official CPU-only PyTorch index. The required packages for the official notebooks are covered by the shared set.

In [ ]:
%cd /content/rea_of_llms_supplementary_code-main
!pip install -r requirements.txt

## 3. Edit Run Parameters Here

This is the main control panel for the notebook. Change these values first, then rerun the cells below.

Conceptually, the methods are not doing the same statistical job:

- direct sampling gives a plain Monte Carlo estimate under the base model `p_M`;
- SMC is used here to sample from a tilted target `p_lambda(x) proportional to p_M(x | c) exp{-lambda phi(x)}` and to discover high-ARI completions;
- the official paper pipeline can use TPS+MBAR for reweighting/probability estimation when its outputs are available.

For now, SMC is **not** reported as an estimator of `P_pM(raw ARI >= x)`. It reports tilted-sample diagnostics and discovery statistics.


In [ ]:
# =========================
# EDIT PARAMETERS HERE
# =========================

MODEL_NAME = "roneneldan/TinyStories-8M"
TOKENIZER_NAME = "EleutherAI/gpt-neo-125M"
PROMPT = "Once upon a time, in a big forest, there lived a rhinoc"

# Reproducibility. For the next phase we will repeat this over many seeds;
# for now, all three methods use this same fixed seed setup.
SEED = 42
PY_SEED = 1234

# Completion and high-ARI diagnostic thresholds.
# TAIL_THRESHOLD_X is used for direct probability and for discovery counts.
# SMC uses the same threshold only as a diagnostic, not as a p_M probability estimate.
MAX_NEW_TOKENS = 100
TAIL_THRESHOLD_X = 8.0
TAIL_COUNT_THRESHOLDS = [6.0, 7.0, 8.0]
ARI_CAP_FOR_SMC_POTENTIAL = 15.0

# Direct ancestral sampling. Increase DIRECT_SAMPLES for a lower-variance
# direct estimate of P(raw ARI >= x).
DIRECT_SAMPLES = 1024
DIRECT_BATCH_SIZE = 32

# Full-sequence SMC. This ends at lambda=-2 and therefore targets the tilted
# distribution, not the base model p_M. We use it for discovery/sampling, not
# for estimating P_pM(raw ARI >= x) in this notebook.
SMC_PARTICLES = 64
SMC_MCMC_STEPS_PER_LEVEL = 5
SMC_LAMBDAS = [0.0, -0.25, -0.5, -1.0, -1.5, -2.0]

# Official paper TPS+MBAR pipeline. The official code is notebook-based and
# unbatched; these values make a longer single-seed run without trying to match
# exact wall-clock compute. Set FORCE_RERUN_OFFICIAL_SAMPLING=False to reuse
# existing official outputs in the selected subdirectory.
RUN_OFFICIAL_SAMPLING_NOTEBOOK = True
FORCE_RERUN_OFFICIAL_SAMPLING = True
OFFICIAL_OUTPUT_SUBDIR = "comparison_single_seed_long"
OFFICIAL_STEPS_PER_BIAS = 50
OFFICIAL_NUM_TRAJECTORIES = 4
OFFICIAL_NUM_UNBIASED_SAMPLES = DIRECT_SAMPLES
OFFICIAL_PREVIEW_COMPLETIONS = 20
OFFICIAL_ARI_BIAS_SCHEDULES = [
    [0.25, 0.5, 1.0, 1.5, 2.0],
    [-0.25, -0.5, -1.0, -1.5, -2.0],
]
OFFICIAL_RUN_LOG_PROBS = False

smc_generated_token_budget = SMC_PARTICLES * MAX_NEW_TOKENS * (1 + (len(SMC_LAMBDAS) - 1) * SMC_MCMC_STEPS_PER_LEVEL)
direct_generated_token_budget = DIRECT_SAMPLES * MAX_NEW_TOKENS
official_tps_transition_budget = OFFICIAL_STEPS_PER_BIAS * OFFICIAL_NUM_TRAJECTORIES * sum(len(schedule) for schedule in OFFICIAL_ARI_BIAS_SCHEDULES)
official_generated_token_budget_approx = (OFFICIAL_NUM_UNBIASED_SAMPLES + OFFICIAL_PREVIEW_COMPLETIONS + official_tps_transition_budget) * MAX_NEW_TOKENS

print("Prompt:", PROMPT)
print("Seed:", SEED)
print("High-ARI diagnostic threshold: raw ARI >=", TAIL_THRESHOLD_X)
print("Direct samples:", DIRECT_SAMPLES)
print("SMC particles:", SMC_PARTICLES)
print("SMC lambdas:", SMC_LAMBDAS)
print("SMC final target lambda:", SMC_LAMBDAS[-1])
print("SMC MCMC steps per level:", SMC_MCMC_STEPS_PER_LEVEL)
print("Official ARI schedules:", OFFICIAL_ARI_BIAS_SCHEDULES)
print("Official steps per bias:", OFFICIAL_STEPS_PER_BIAS)
print("Official trajectories:", OFFICIAL_NUM_TRAJECTORIES)
print("Direct generated completion-token transitions:", direct_generated_token_budget)
print("Approximate SMC generated completion-token transitions:", smc_generated_token_budget)
print("Approximate official generated completion-token transitions:", official_generated_token_budget_approx)


## 4. Load Model and Shared Utilities

The model is loaded once and reused by the direct and SMC cells. Use a GPU runtime in Colab if available.

In [ ]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.special import logsumexp
from textstat import automated_readability_index
from tqdm.auto import tqdm

from rea_llms.models import load_model_and_tokenizer, set_seed
from rea_llms.observables import create_observable

set_seed(SEED)
random.seed(PY_SEED)
np.random.seed(SEED)

model, tokenizer = load_model_and_tokenizer(
    MODEL_NAME,
    TOKENIZER_NAME,
    device="auto",
)
observable = create_observable("ari", mode="capped", cap=ARI_CAP_FOR_SMC_POTENTIAL, backend="exact_cached")
model_device = next(model.parameters()).device
pad_token_id = getattr(tokenizer, "pad_token_id", None)
print("Model device:", model_device)

## 5. Direct Sampling Estimate With Progress Bar

This estimates `P_pM(ARI >= x)` by ordinary direct ancestral sampling.

In [ ]:
def direct_sampling_with_progress():
    set_seed(SEED)
    prompt_ids = tokenizer(PROMPT, return_tensors="pt")["input_ids"].to(model_device)
    prompt_token_count = int(prompt_ids.shape[1])
    rows = []
    start = time.perf_counter()

    for offset in tqdm(range(0, DIRECT_SAMPLES, DIRECT_BATCH_SIZE), desc="Direct sampling"):
        current_batch = min(DIRECT_BATCH_SIZE, DIRECT_SAMPLES - offset)
        input_ids = prompt_ids.repeat(current_batch, 1)
        with torch.no_grad():
            generated = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                top_k=None,
                temperature=1.0,
                eos_token_id=None,
                pad_token_id=pad_token_id,
            )
        completion_ids = generated[:, prompt_token_count:].detach().cpu().tolist()
        completions = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)
        for local_idx, completion in enumerate(completions):
            score = observable.score_completion(PROMPT, completion)
            raw_ari = float(score.metadata["raw_ari"])
            capped_ari = float(score.metadata["capped_ari"])
            rows.append({
                "method": "direct",
                "sample_id": offset + local_idx,
                "completion": completion,
                "raw_ari": raw_ari,
                "phi": capped_ari,
                "weight": 1.0,
            })

    elapsed = time.perf_counter() - start
    df = pd.DataFrame(rows)
    prob = float((df["raw_ari"] >= TAIL_THRESHOLD_X).mean()) if len(df) else float("nan")
    print(f"Direct P(raw ARI >= {TAIL_THRESHOLD_X}) = {prob:.6g}")
    print(f"Direct runtime seconds = {elapsed:.3f}")
    return df, prob, elapsed

direct_df, direct_tail_probability, direct_runtime_seconds = direct_sampling_with_progress()
direct_df.head()


## 6. Full-Sequence SMC Tilted Sampler With Progress Bars

This runs the SMC implementation from this repository, but keeps the control loop visible in the notebook so progress bars can show lambda levels and MCMC mutation steps.

The final SMC target is the tilted distribution at `SMC_LAMBDAS[-1]`. The high-ARI count printed below is a discovery/statistics diagnostic under the tilted particle system; it is **not** a base-model rare-event probability estimate.


In [ ]:
from full_sequence_smc_experiment.smc_sampler import (
    apply_mcmc_mutation_kernel,
    build_weighted_particle_samples,
    ess_over_n_from_log_weights,
    make_level_diagnostic,
    normalize_log_weights,
    particle_phi_values,
    rare_event_mask,
    sample_particles_from_base_model,
    update_log_weights_for_lambda_step,
)


def run_smc_with_progress():
    set_seed(SEED)
    rng = np.random.default_rng(SEED)
    torch_generator = torch.Generator(device=model_device).manual_seed(SEED)
    prompt_ids = tokenizer(PROMPT, return_tensors="pt")["input_ids"][0].detach().cpu().tolist()
    prompt_token_count = len(prompt_ids)
    start = time.perf_counter()

    particles = sample_particles_from_base_model(
        model,
        tokenizer,
        observable,
        prompt=PROMPT,
        prompt_ids=prompt_ids,
        prompt_token_count=prompt_token_count,
        num_particles=SMC_PARTICLES,
        max_new_tokens=MAX_NEW_TOKENS,
        generator=torch_generator,
        pad_token_id=pad_token_id,
    )
    log_weights = np.zeros(SMC_PARTICLES, dtype=np.float64)
    level_diagnostics = [
        make_level_diagnostic(
            level=0,
            current_lambda=float(SMC_LAMBDAS[0]),
            previous_lambda=None,
            ess_over_n=1.0,
            accepted=0,
            proposals=0,
            mean_phi=float(np.mean(particle_phi_values(particles))),
        )
    ]
    total_accepted = 0
    total_proposals = 0

    for level in tqdm(range(1, len(SMC_LAMBDAS)), desc="SMC lambda levels"):
        previous_lambda = float(SMC_LAMBDAS[level - 1])
        current_lambda = float(SMC_LAMBDAS[level])
        level_accepted = 0
        level_proposals = 0

        for _ in tqdm(range(SMC_MCMC_STEPS_PER_LEVEL), desc=f"MCMC at lambda {previous_lambda:g}", leave=False):
            mutation = apply_mcmc_mutation_kernel(
                particles,
                model,
                tokenizer,
                observable,
                prompt=PROMPT,
                prompt_ids=prompt_ids,
                prompt_token_count=prompt_token_count,
                target_lambda=previous_lambda,
                mcmc_steps=1,
                max_new_tokens=MAX_NEW_TOKENS,
                torch_generator=torch_generator,
                rng=rng,
                pad_token_id=pad_token_id,
            )
            particles = mutation.particles
            level_accepted += mutation.accepted
            level_proposals += mutation.proposals

        total_accepted += level_accepted
        total_proposals += level_proposals
        log_weights, phi_values = update_log_weights_for_lambda_step(
            log_weights,
            particles,
            previous_lambda=previous_lambda,
            current_lambda=current_lambda,
        )
        level_diagnostics.append(
            make_level_diagnostic(
                level=level,
                current_lambda=current_lambda,
                previous_lambda=previous_lambda,
                ess_over_n=ess_over_n_from_log_weights(log_weights),
                accepted=level_accepted,
                proposals=level_proposals,
                mean_phi=float(np.mean(phi_values)),
            )
        )

    normalized_weights = np.exp(normalize_log_weights(log_weights))
    final_phi = particle_phi_values(particles)
    raw_ari_values = np.asarray([
        float(particle.metadata.get("raw_ari", particle.phi))
        for particle in particles
    ], dtype=np.float64)
    rare_events = rare_event_mask(raw_ari_values, TAIL_THRESHOLD_X, "ge")
    samples = build_weighted_particle_samples(
        particles,
        log_weights=log_weights,
        normalized_weights=normalized_weights,
        rare_events=rare_events,
    )
    elapsed = time.perf_counter() - start
    df = pd.DataFrame(samples)
    df["weight"] = df["normalized_weight"].astype(float)
    df["phi"] = df["phi"].astype(float)
    df["raw_ari"] = df["raw_ari"].astype(float)
    tilted_tail_mass = float(df.loc[df["raw_ari"] >= TAIL_THRESHOLD_X, "weight"].sum())
    diagnostics_df = pd.DataFrame(level_diagnostics)
    print(f"SMC weighted tilted mass with raw ARI >= {TAIL_THRESHOLD_X} = {tilted_tail_mass:.6g}")
    print("This is under the final tilted target, not a p_M probability estimate.")
    print(f"SMC runtime seconds = {elapsed:.3f}")
    print(f"SMC total acceptance rate = {total_accepted / total_proposals if total_proposals else 0.0:.6g}")
    return df, tilted_tail_mass, elapsed, diagnostics_df

smc_df, smc_tilted_tail_mass, smc_runtime_seconds, smc_diagnostics_df = run_smc_with_progress()
smc_diagnostics_df


## 7. Run the Official Paper Code With the Same Seed

The official repository is included as a Git submodule at `external/rea_of_llms_official`. Its original pipeline is notebook-based:

- `sampling_examples.ipynb`: direct sampling and annealed TPS examples;
- `reweighting_example.ipynb`: burn-in, Gelman-Rubin, MBAR, bootstrap intervals.

The next cell creates a temporary configured copy of `sampling_examples.ipynb` inside `outputs/` and runs that copy. This keeps the official submodule unchanged while letting us use the same prompt, seed, completion length and larger single-seed settings selected above.

PyMBAR may print warnings about JAX 64-bit mode and time-series assumptions; those are informational warnings, not failures.


In [ ]:
from IPython.display import display, Markdown

OFFICIAL_DIR = REPO_DIR / "external" / "rea_of_llms_official"
display(Markdown(f"Official code directory: `{OFFICIAL_DIR}`"))
display(Markdown(f"Official sampling notebook: `{OFFICIAL_DIR / 'sampling_examples.ipynb'}`"))
display(Markdown(f"Official reweighting notebook: `{OFFICIAL_DIR / 'reweighting_example.ipynb'}`"))

print("Official notebook files:")
!ls -lh external/rea_of_llms_official/*.ipynb

In [ ]:
import re

configured_official_notebook = REPO_DIR / "outputs" / "official_sampling_configured.ipynb"
official_example_dir = OFFICIAL_DIR / "data" / OFFICIAL_OUTPUT_SUBDIR
official_required_outputs = [
    official_example_dir / "ari_trajectories.json",
    official_example_dir / "ari_unbiased_samples.json",
]
official_outputs_exist = all(path.exists() for path in official_required_outputs)


def write_configured_official_sampling_notebook():
    source_path = OFFICIAL_DIR / "sampling_examples.ipynb"
    configured_official_notebook.parent.mkdir(parents=True, exist_ok=True)
    official_nb = json.loads(source_path.read_text(encoding="utf-8"))

    for cell in official_nb["cells"]:
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        source = source.replace("import os\n", "import os\nfrom tqdm.auto import tqdm\n")
        source = re.sub(r'^prompt = .*$', f"prompt = {PROMPT!r}", source, flags=re.MULTILINE)
        source = re.sub(r'^TORCH_SEED = .*$', f"TORCH_SEED = {int(SEED)}", source, flags=re.MULTILINE)
        source = re.sub(r'^PY_SEED = .*$', f"PY_SEED = {int(PY_SEED)}", source, flags=re.MULTILINE)
        source = re.sub(r'^NUM_COMPLETIONS = .*$', f"NUM_COMPLETIONS = {int(OFFICIAL_PREVIEW_COMPLETIONS)}", source, flags=re.MULTILINE)
        source = re.sub(r'^MAX_GEN_TOKENS = .*$', f"MAX_GEN_TOKENS = {int(MAX_NEW_TOKENS)}", source, flags=re.MULTILINE)
        source = re.sub(
            r'^DATA_SAVE_DIR = .*$',
            f'DATA_SAVE_DIR = os.path.join("data", {OFFICIAL_OUTPUT_SUBDIR!r})',
            source,
            flags=re.MULTILINE,
        )
        source = source.replace(
            "for _ in range(num_completions):",
            "for _ in tqdm(range(num_completions), desc='Official direct sampling', leave=False):",
        )
        source = re.sub(r'^steps_per_bias = .*$', f"steps_per_bias = {int(OFFICIAL_STEPS_PER_BIAS)}", source, flags=re.MULTILINE)
        source = re.sub(r'^num_trajectories = .*$', f"num_trajectories = {int(OFFICIAL_NUM_TRAJECTORIES)}", source, flags=re.MULTILINE)
        source = re.sub(r'^num_unbiased_samples = .*$', f"num_unbiased_samples = {int(OFFICIAL_NUM_UNBIASED_SAMPLES)}", source, flags=re.MULTILINE)
        old_bias_block = 'both_obs_biases = {"ari": [[0.1, 0.2, 0.3], [-0.1, -0.2, -0.3]],\n              "log_probs": [[[0.01, 0.02, 0.03], [-0.01, -0.02, -0.03]]]}'
        new_bias_block = (
            f'both_obs_biases = {{"ari": {repr(OFFICIAL_ARI_BIAS_SCHEDULES)},\n'
            f'              "log_probs": []}}'
        )
        source = source.replace(old_bias_block, new_bias_block)
        official_observables = '["ari", "log_probs"]' if OFFICIAL_RUN_LOG_PROBS else '["ari"]'
        source = source.replace(
            'for obs_name in ["ari", "log_probs"]:',
            f'for obs_name in tqdm({official_observables}, desc="Official observables"):',
        )
        source = source.replace(
            'for biases in all_biases:',
            'for biases in tqdm(all_biases, desc=f"Official schedules for {obs_name}", leave=False):',
        )
        source = source.replace(
            'for num_traj in range(num_trajectories):',
            'for num_traj in tqdm(range(num_trajectories), desc=f"Official TPS {obs_name} {biases}", leave=False):',
        )
        cell["source"] = [line + "\n" for line in source.rstrip("\n").split("\n")]

    configured_official_notebook.write_text(json.dumps(official_nb, indent=1), encoding="utf-8")
    return configured_official_notebook


if RUN_OFFICIAL_SAMPLING_NOTEBOOK:
    if official_outputs_exist and not FORCE_RERUN_OFFICIAL_SAMPLING:
        print("Official ARI outputs already exist; reusing them.")
    else:
        configured_path = write_configured_official_sampling_notebook()
        print("Running configured official sampling notebook:", configured_path)
        print("Official output directory:", official_example_dir)
        ip = get_ipython()
        ip.run_line_magic("cd", str(OFFICIAL_DIR))
        ip.run_line_magic("run", str(configured_path))
        ip.run_line_magic("cd", str(REPO_DIR))
        missing = [str(path) for path in official_required_outputs if not path.exists()]
        if missing:
            raise FileNotFoundError("The official notebook finished, but these expected files are still missing: " + ", ".join(missing))
        print("Official ARI outputs created successfully.")
else:
    print("Official sampling is disabled. Set RUN_OFFICIAL_SAMPLING_NOTEBOOK = True to execute it.")


## 8. Paper-Pipeline Tail Probability From Official TPS + MBAR Outputs

This cell estimates `P_pM(phi >= x)` from the official small-example output files. It mirrors the official reweighting notebook: burn-in, Gelman-Rubin filtering, adding unbiased samples, MBAR normalizers, and self-normalized weighted tail probability.

In [ ]:
import pymbar


def _official_group_by_bias(trajectories, biases, steps_per_bias):
    all_biases = []
    grouped = []
    for biases_for_schedule, trajs_per_schedule in zip(biases, trajectories):
        cum_steps = 0
        for bias in biases_for_schedule:
            if bias not in all_biases:
                all_biases.append(bias)
                grouped.append([])
            index = all_biases.index(bias)
            for traj in trajs_per_schedule:
                grouped[index].append(traj[cum_steps: cum_steps + steps_per_bias])
            cum_steps += steps_per_bias
    sorted_indices = np.argsort(all_biases)
    sorted_biases = [float(all_biases[i]) for i in sorted_indices]
    sorted_grouped = [np.asarray(grouped[i], dtype=np.float64) for i in sorted_indices]
    return sorted_biases, sorted_grouped


def _official_apply_burnin(trajectories, biases, steps_per_bias, burnin=0.1):
    burnin_steps = int(steps_per_bias * burnin)
    new_steps = steps_per_bias - burnin_steps
    new_trajectories = []
    for biases_for_schedule, trajs_per_schedule in zip(biases, trajectories):
        new_trajs = []
        for traj in trajs_per_schedule:
            kept = []
            for i in range(len(biases_for_schedule)):
                kept += traj[i * steps_per_bias + burnin_steps: (i + 1) * steps_per_bias]
            new_trajs.append(kept)
        new_trajectories.append(new_trajs)
    return new_trajectories, new_steps


def _official_gelman_rubin_scores(trajectories, biases, steps_per_bias, cutoff=1.1):
    bias_values, grouped = _official_group_by_bias(trajectories, biases, steps_per_bias)
    scores_by_bias = {}
    gr_values = {}
    for bias, trajs_per_bias in zip(bias_values, grouped):
        j, L = trajs_per_bias.shape
        if j < 2 or L < 2:
            gr_values[bias] = float("inf")
            continue
        means = np.mean(trajs_per_bias, axis=1)
        mean_of_means = np.mean(means)
        B = L / (j - 1) * np.sum((means - mean_of_means) ** 2)
        W = 1 / j * np.sum(np.var(trajs_per_bias, axis=1, ddof=1))
        R_hat = float(np.sqrt((((L - 1) / L) * W + B / L) / W)) if W > 0 else float("inf")
        gr_values[bias] = R_hat
        if R_hat < cutoff:
            scores_by_bias[bias] = trajs_per_bias.flatten().tolist()
    return scores_by_bias, gr_values


def estimate_official_tps_mbar_tail_probability():
    example_dir = OFFICIAL_DIR / "data" / OFFICIAL_OUTPUT_SUBDIR
    traj_path = example_dir / "ari_trajectories.json"
    unbiased_path = example_dir / "ari_unbiased_samples.json"
    if not traj_path.exists() or not unbiased_path.exists():
        raise FileNotFoundError(
            "Official ARI example outputs not found. Run the previous official sampling cell first, with RUN_OFFICIAL_SAMPLING_NOTEBOOK = True."
        )

    trajectories, biases, steps_per_bias = json.loads(traj_path.read_text(encoding="utf-8"))
    unbiased_scores = json.loads(unbiased_path.read_text(encoding="utf-8"))
    burned_trajectories, burned_steps = _official_apply_burnin(trajectories, biases, steps_per_bias, burnin=0.1)
    scores_by_bias, gr_values = _official_gelman_rubin_scores(burned_trajectories, biases, burned_steps, cutoff=1.1)

    num_trajs_per_schedule = len(trajectories[0])
    unbiased_samples_for_biased = int(steps_per_bias * num_trajs_per_schedule / 2)
    scores_by_bias.setdefault(0.0, [])
    scores_by_bias[0.0].extend(unbiased_scores[:unbiased_samples_for_biased])

    bias_values = sorted(scores_by_bias)
    all_scores = np.concatenate([np.asarray(scores_by_bias[b], dtype=np.float64) for b in bias_values])
    Ns = np.asarray([len(scores_by_bias[b]) for b in bias_values], dtype=np.int64)
    u_kn = np.outer(np.asarray(bias_values, dtype=np.float64), all_scores)

    mbar = pymbar.MBAR(u_kn, Ns)
    delta_f = mbar.compute_free_energy_differences()["Delta_f"]
    zero_index = bias_values.index(0.0)
    log_Zs = (-delta_f[zero_index]).astype(float)

    score_rows = []
    for i, bias in enumerate(bias_values):
        scores = np.asarray(scores_by_bias[bias], dtype=np.float64)
        these_weights = np.exp(log_Zs[i] + bias * scores)
        for score, weight in zip(scores, these_weights):
            score_rows.append({
                "method": "official_tps_mbar",
                "phi": float(score),
                "raw_ari": float(score),
                "weight": float(weight),
                "bias": float(bias),
            })

    official_df = pd.DataFrame(score_rows)
    official_df["normalized_weight"] = official_df["weight"] / official_df["weight"].sum()
    probability = float(official_df.loc[official_df["raw_ari"] >= TAIL_THRESHOLD_X, "normalized_weight"].sum())
    print(f"Official TPS+MBAR P(raw ARI >= {TAIL_THRESHOLD_X}) = {probability:.6g}")
    print("GR values:", gr_values)
    return official_df, probability, pd.DataFrame({"bias": list(gr_values), "gelman_rubin": list(gr_values.values())})

try:
    official_df, official_tail_probability, official_gr_df = estimate_official_tps_mbar_tail_probability()
    display(official_gr_df)
except FileNotFoundError as exc:
    official_df = pd.DataFrame()
    official_tail_probability = float("nan")
    official_gr_df = pd.DataFrame()
    print(exc)


## 9. Compare Probability and ARI Statistics

The table keeps probability-estimation and sampling/discovery roles separate.

- `direct` reports an ordinary base-model Monte Carlo probability estimate.
- `official_tps_mbar` reports a base-model probability estimate when the MBAR outputs are available.
- `smc_tilted_lambda_final` reports weighted statistics under its final tilted target and observed high-ARI discovery counts. Its `base_probability_estimate_raw_ari_ge_x` column is intentionally left empty.


In [ ]:
def weighted_mean(values, weights):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    if len(values) == 0 or float(np.sum(weights)) == 0.0:
        return float("nan")
    weights = weights / np.sum(weights)
    return float(np.sum(weights * values))


def weighted_variance(values, weights):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    if len(values) == 0 or float(np.sum(weights)) == 0.0:
        return float("nan")
    weights = weights / np.sum(weights)
    mean = np.sum(weights * values)
    return float(np.sum(weights * (values - mean) ** 2))


def summarize_method(
    name,
    df,
    *,
    base_probability_estimate=None,
    tilted_tail_mass=None,
    runtime_seconds=None,
    generated_token_budget=None,
    use_weights=False,
    target_description="p_M",
):
    row = {
        "method": name,
        "target_description": target_description,
        "base_probability_estimate_raw_ari_ge_x": base_probability_estimate,
        "tilted_weighted_mass_raw_ari_ge_x": tilted_tail_mass,
        "runtime_seconds": runtime_seconds,
        "generated_token_budget": generated_token_budget,
    }
    if df is None or len(df) == 0:
        return row

    values = df["raw_ari"].astype(float).to_numpy()
    weights = df["weight"].astype(float).to_numpy() if use_weights and "weight" in df else np.ones(len(df), dtype=np.float64)
    row.update({
        "num_samples_or_particles": int(len(df)),
        "max_ari_observed": float(np.max(values)),
        "mean_ari": weighted_mean(values, weights),
        "median_ari_observed": float(np.median(values)),
        "variance_ari": weighted_variance(values, weights),
    })
    for threshold in TAIL_COUNT_THRESHOLDS:
        row[f"observed_count_ari_ge_{threshold:g}"] = int(np.sum(values >= threshold))
    return row

comparison_rows = [
    summarize_method(
        "direct",
        direct_df,
        base_probability_estimate=direct_tail_probability,
        runtime_seconds=direct_runtime_seconds,
        generated_token_budget=direct_generated_token_budget,
        use_weights=False,
        target_description="base model p_M",
    ),
    summarize_method(
        "smc_tilted_lambda_final",
        smc_df,
        tilted_tail_mass=smc_tilted_tail_mass,
        runtime_seconds=smc_runtime_seconds,
        generated_token_budget=smc_generated_token_budget,
        use_weights=True,
        target_description=f"tilted target p_lambda, lambda={SMC_LAMBDAS[-1]}",
    ),
]
if len(official_df):
    comparison_rows.append(
        summarize_method(
            "official_tps_mbar",
            official_df,
            base_probability_estimate=official_tail_probability,
            runtime_seconds=None,
            generated_token_budget=official_generated_token_budget_approx,
            use_weights=True,
            target_description="base model p_M via TPS+MBAR reweighting",
        )
    )
else:
    comparison_rows.append({
        "method": "official_tps_mbar",
        "target_description": "base model p_M via TPS+MBAR reweighting",
        "base_probability_estimate_raw_ari_ge_x": float("nan"),
        "note": "Run official sampling notebook first.",
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
Path("outputs").mkdir(parents=True, exist_ok=True)
comparison_df.to_csv("outputs/comparison_probability_and_ari_stats.csv", index=False)
print("Saved: outputs/comparison_probability_and_ari_stats.csv")


## 10. Box Plots and Tail-Count Visuals

These plots show the empirical spread of raw ARI values seen by each method in this single-seed run. For SMC, read the box plot as the distribution of the final tilted particle system, not as a base-model probability estimate.


In [ ]:
import matplotlib.pyplot as plt

plot_frames = []
for name, df in [("direct", direct_df), ("smc", smc_df), ("official_tps_mbar", official_df)]:
    if df is not None and len(df):
        tmp = df[["raw_ari"]].copy()
        tmp["method"] = name
        plot_frames.append(tmp)

plot_df = pd.concat(plot_frames, ignore_index=True) if plot_frames else pd.DataFrame(columns=["method", "raw_ari"])
if len(plot_df):
    methods = list(plot_df["method"].unique())
    grouped = [plot_df.loc[plot_df["method"] == method, "raw_ari"].astype(float).to_numpy() for method in methods]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].boxplot(grouped, labels=methods, showmeans=True)
    axes[0].axhline(TAIL_THRESHOLD_X, color="red", linestyle="--", linewidth=1, label=f"x={TAIL_THRESHOLD_X}")
    axes[0].set_title("Raw ARI Distribution by Method")
    axes[0].set_ylabel("raw ARI")
    axes[0].tick_params(axis="x", rotation=20)
    axes[0].legend()

    count_rows = []
    for method in methods:
        values = plot_df.loc[plot_df["method"] == method, "raw_ari"].astype(float).to_numpy()
        for threshold in TAIL_COUNT_THRESHOLDS:
            count_rows.append({"method": method, "threshold": threshold, "count": int(np.sum(values >= threshold))})
    count_df = pd.DataFrame(count_rows)
    x_positions = np.arange(len(TAIL_COUNT_THRESHOLDS))
    width = 0.8 / max(len(methods), 1)
    for offset, method in enumerate(methods):
        counts = [int(count_df[(count_df["method"] == method) & (count_df["threshold"] == threshold)]["count"].iloc[0]) for threshold in TAIL_COUNT_THRESHOLDS]
        axes[1].bar(x_positions + offset * width, counts, width=width, label=method)
    axes[1].set_xticks(x_positions + width * (len(methods) - 1) / 2)
    axes[1].set_xticklabels([f"ARI >= {t:g}" for t in TAIL_COUNT_THRESHOLDS])
    axes[1].set_title("Observed Tail Counts")
    axes[1].set_ylabel("count")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    display(count_df)
else:
    print("No samples available for plotting yet.")
